# Day 2 · Validation and tuning | اليوم الثاني

**Naif Faraj AlDosari · SDA-DSC-211**

Day 1 put XGBoost ahead on Average Precision by 0.0080, but 1,226 customers appeared on both sides of the random split. Day 2 tests the validation setup with LightGBM as the lab's common model. The Day 1 choice stays provisional until the same models can be compared under the stronger split.

### Set up the lab

The lab checks the package versions and downloads fixed revisions of the course files. I use the default FAST settings on CPU with seed `211`. [Day 2 guide](https://github.com/almiyead-rgb/sda-dsc-211-student-template/blob/main/DAY2_GUIDE.md) · [Course notebook](https://github.com/almiyead-rgb/sda-dsc-211-student-template/blob/fc25fd3ba9f8a1e596e4b1d991ee44743c375b0a/notebooks/02_validation_tuning.ipynb)

<table style="width:100%; border-collapse:collapse;">
<tr>
<td width="50%" valign="top" dir="ltr" style="padding:14px; border:1px solid #d0d7de;">
<h2>Day 2 · Validate honestly before tuning</h2>
<p><strong>SDA-DSC-211 · Tamweel Lite · 60-minute lab · Free CPU</strong></p>
<p>Audit feature availability, remove post-outcome leakage and exact duplicate events, then evaluate later requests with mature labels while excluding each validation fold's customers from its training fold. Freeze a bounded search on a separate historical customer cohort before reading outer-validation results.</p>
<p><strong>Deliverables:</strong> leakage and fold audits, bounded-search history, frozen parameters, validation report and summary, out-of-fold predictions and coverage, provenance, figures, run metadata and your written interpretation.</p>
<p><strong>Scientific boundary:</strong> all records are synthetic. The random protocols are negative controls, not deployment evidence. The forward folds estimate a particular later-period, unseen-customer scenario; they are not a final independent test and do not prove fairness, calibration or operational value.</p>
<table><tr><th>Minutes</th><th>Your task</th></tr><tr><td>0–8</td><td>Prepare the environment and read the data contract</td></tr><tr><td>8–18</td><td>Audit leakage and duplicate events</td></tr><tr><td>18–30</td><td>Inspect time, customer and label-maturity boundaries</td></tr><tr><td>30–45</td><td>Run the bounded search on the reserved cohort</td></tr><tr><td>45–53</td><td>Compare means, variation and coverage</td></tr><tr><td>53–60</td><td>Write your reasoning and export evidence</td></tr></table>
</td>
<td width="50%" valign="top" dir="rtl" style="padding:14px; border:1px solid #d0d7de;">
<h2>اليوم الثاني · تحقّق بصدق قبل ضبط النموذج</h2>
<p><strong>SDA-DSC-211 · Tamweel Lite · لاب 60 دقيقة · CPU مجاني</strong></p>
<p>دقّق زمن إتاحة الخصائص، واحذف تسرب ما بعد النتيجة والنسخ المتطابقة من الحدث، ثم قيّم الطلبات اللاحقة بعد نضج أهدافها مع استبعاد عملاء كل طية تحقق من تدريب الطية نفسها. جمّد بحثًا محدودًا على مجموعة تاريخية منفصلة قبل قراءة نتائج التحقق الخارجي.</p>
<p><strong>المخرجات:</strong> تدقيق التسرب والطيات، وسجل البحث المحدود، والإعدادات المجمدة، وتقرير التحقق وملخصه، وتنبؤات OOF وتغطيتها، ومصدر التدريب، والرسوم، وبيانات التشغيل، وتفسيرك المكتوب.</p>
<p><strong>حد علمي:</strong> جميع السجلات اصطناعية. البروتوكولات العشوائية ضوابط سلبية وليست دليل نشر. تقيس الطيات الزمنية سيناريو محددًا لطلبات لاحقة من عملاء غير موجودين في تدريب الطية؛ وليست اختبارًا نهائيًا مستقلًا، ولا تثبت العدالة أو المعايرة أو القيمة التشغيلية.</p>
<table><tr><th>الدقائق</th><th>مهمتك</th></tr><tr><td>0–8</td><td>جهّز البيئة واقرأ عقد البيانات</td></tr><tr><td>8–18</td><td>دقّق التسرب والطلبات المكررة</td></tr><tr><td>18–30</td><td>افحص حدود الزمن والعملاء ونضج الهدف</td></tr><tr><td>30–45</td><td>نفّذ البحث المحدود على المجموعة المحجوزة</td></tr><tr><td>45–53</td><td>قارن المتوسط والتشتت والتغطية</td></tr><tr><td>53–60</td><td>اكتب تفسيرك وصدّر الأدلة</td></tr></table>
</td>
</tr>
</table>

<table style="width:100%; border-collapse:collapse;">
<tr>
<td width="50%" valign="top" dir="ltr" style="padding:14px; border:1px solid #d0d7de;">
<h2>Official daily requirement</h2>
<p>Run all cells, compare the generated files with EXPECTED_RESULTS.md, then answer only the single question at the end of this notebook in your repository README. Legacy reflection variables inside code cells are compatibility fields; leave them unchanged.</p>
</td>
<td width="50%" valign="top" dir="rtl" style="padding:14px; border:1px solid #d0d7de;">
<h2>المطلوب اليومي المعتمد</h2>
<p>شغّل جميع الخلايا، وقارن الملفات الناتجة بصفحة EXPECTED_RESULTS.md، ثم أجب فقط عن السؤال الواحد في نهاية هذا الدفتر داخل README في مستودعك. حقول الانعكاس القديمة داخل خلايا الكود مخصصة للتوافق؛ اتركها دون تعديل.</p>
</td>
</tr>
</table>

<table style="width:100%; border-collapse:collapse;">
<tr>
<td width="50%" valign="top" dir="ltr" style="padding:14px; border:1px solid #d0d7de;">
<h2>1. Prepare a controlled environment</h2>
<p>Open a fresh notebook session, select the free CPU runtime and choose <strong>Runtime → Run all</strong>. Setup verifies pinned library versions, the data manifest and SHA-256 hashes before modeling.</p>
<ul><li><code>FAST_MODE=True</code>: two CPU threads and a ceiling of 200 trees; use this first.</li><li><code>FULL_MODE=True</code>: optional ceiling of 600 trees.</li><li>Both modes retain at most eight sequential trials, three search folds and a 120-second cooperative search budget.</li><li>No GPU, API key, Drive mount or paid service is required.</li></ul>
<p><strong>Pass condition:</strong> the cell prints <code>Day 2 setup verified</code>. Do not bypass checksum or imported-version failures; restart the session and run from the beginning.</p>
</td>
<td width="50%" valign="top" dir="rtl" style="padding:14px; border:1px solid #d0d7de;">
<h2>1. جهّز بيئة تشغيل محكومة</h2>
<p>افتح جلسة دفتر جديدة، واختر بيئة CPU المجانية، ثم نفّذ <strong>Runtime → Run all</strong>. يتحقق الإعداد من إصدارات المكتبات المثبتة وبيان البيانات وبصمات SHA-256 قبل بدء النمذجة.</p>
<ul><li><code>FAST_MODE=True</code>: خيطا CPU وسقف 200 شجرة؛ ابدأ به.</li><li><code>FULL_MODE=True</code>: سقف اختياري يبلغ 600 شجرة.</li><li>يحتفظ الوضعان بحد أقصى قدره ثماني تجارب متتابعة وثلاث طيات للبحث وميزانية بحث تعاونية قدرها 120 ثانية.</li><li>لا تحتاج إلى GPU أو مفتاح API أو ربط Drive أو خدمة مدفوعة.</li></ul>
<p><strong>شرط الاجتياز:</strong> ظهور <code>Day 2 setup verified</code>. لا تتجاوز فشل البصمة أو اختلاف إصدار مستورد؛ أعد تشغيل الجلسة ثم شغّل الدفتر من البداية.</p>
</td>
</tr>
</table>

In [1]:
# Verify the released inputs before comparing any scores.
from pathlib import Path
import hashlib, importlib.util, os, runpy, sys, time, urllib.request, urllib.error

RUN_STARTED = time.perf_counter()
COURSE_REVISION = "fe0c0204e6076a7ac2139b7336485a097343fb8a"
MANIFEST_SHA256 = "86d15e813caf833b7d86512808176388d4123dc3c01c6525ee849a76e8322c59"
SETUP_SHA256 = "79608f8f9559d9e6eb09d51ee541ee12d7e9262ac57d1ca3a948c8ecb14f0f35"
IN_COLAB = importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None
ROOT = Path("/content/tamweel") if IN_COLAB else Path(os.environ.get("TAMWEEL_PROJECT_DIR", ".")).resolve()
setup_path = ROOT / "scripts/setup_colab.py"
setup_path.parent.mkdir(parents=True, exist_ok=True)
if not setup_path.exists() or hashlib.sha256(setup_path.read_bytes()).hexdigest() != SETUP_SHA256:
    setup_url = f"https://raw.githubusercontent.com/almiyead-rgb/sda-dsc-211-student-template/{COURSE_REVISION}/scripts/setup_colab.py"
    for attempt in range(3):
        try:
            with urllib.request.urlopen(setup_url, timeout=45) as response:
                setup_bytes = response.read(200000)
            if hashlib.sha256(setup_bytes).hexdigest() != SETUP_SHA256:
                raise ValueError("Setup checksum mismatch. Reopen the released notebook.")
            setup_path.write_bytes(setup_bytes)
            break
        except (urllib.error.URLError, TimeoutError):
            if attempt == 2:
                raise RuntimeError("Download unavailable. Retry setup or upload the unchanged project files; see READINESS_GUIDE.")
            time.sleep(attempt + 1)
environment = runpy.run_path(str(setup_path))["prepare"](ROOT, COURSE_REVISION, MANIFEST_SHA256)
FAST_MODE, FULL_MODE, SEED, N_JOBS = True, False, 211, 2

from setup_colab import fetch_verified
LAB_REVISION = "1e1da4acbee8941bef07245b259fb6c27ced4ad7"
SUPPORT_HASHES = {'scripts/day2_validation.py': 'd2fd0353330c3629bdaed6c570528792955fce232540a85d33532a9432522f43', 'data/day2_optuna_example.csv': 'a59000eee4caced319735d237b8d121b2e0c7384acce93392d214ddbfe2f5b93', 'data/day2_optuna_example.json': 'a91a4d3a39ecd9cbab9bf902aaf5e2e060d4e21bcc93793057d5250aa002802c'}
for relative, sha256 in SUPPORT_HASHES.items():
    fetch_verified(
        f"https://raw.githubusercontent.com/almiyead-rgb/sda-dsc-211-student-template/{LAB_REVISION}/{relative}",
        ROOT / relative, sha256)
print("Day 2 setup verified | إعداد اليوم الثاني جاهز")

Package                 Version
numpy                   2.1.3
pandas                  2.2.3
scipy                   1.16.3
scikit-learn            1.6.1
matplotlib              3.10.0
xgboost                 3.4.1
lightgbm                4.6.0
shap                    0.52.0
numba                   0.61.2
optuna                  4.5.0
Python 3.12.13 | CPU | seed=211 | n_jobs=2 | FAST_MODE=True
Verified course files. Next: run the data and compatibility checks.


Day 2 setup verified | إعداد اليوم الثاني جاهز


<table style="width:100%; border-collapse:collapse;">
<tr>
<td width="50%" valign="top" dir="ltr" style="padding:14px; border:1px solid #d0d7de;">
<h2>2. Audit information availability and duplicates</h2>
<p>The target <code>default_within_90d</code> records whether a default event occurs within 90 days after the application. It does not mean an account reaches exactly 90 days past due.</p>
<p>Use the feature dictionary's <code>available_at</code> field. A predictor is eligible only when its information exists at application time. A strong correlation alone does not prove leakage.</p>
<table><tr><th>Risk</th><th>Guard</th></tr><tr><td>Target leakage</td><td>Remove fields describing the later outcome</td></tr><tr><td>Preprocessing leakage</td><td>Learn imputation from training rows only</td></tr><tr><td>Time leakage</td><td>Train only on applications whose labels are mature before validation starts</td></tr><tr><td>Customer leakage</td><td>Exclude validation customers from the same fold's training rows</td></tr><tr><td>Duplicate leakage</td><td>Deduplicate identical events and reject conflicting copies</td></tr></table>
<p>The dirty training copy is read to demonstrate the audit. Challenge data is not used for model selection or comparison in this lab.</p>
</td>
<td width="50%" valign="top" dir="rtl" style="padding:14px; border:1px solid #d0d7de;">
<h2>2. دقّق إتاحة المعلومات والتكرار</h2>
<p>يسجل الهدف <code>default_within_90d</code> ما إذا وقع حدث تعثر خلال 90 يومًا بعد الطلب. ولا يعني وصول الحساب إلى 90 يوم تأخر بالضبط.</p>
<p>استخدم حقل <code>available_at</code> في قاموس الخصائص. لا تُقبل الخاصية إلا إذا كانت معلومتها موجودة وقت تقديم الطلب. الارتباط القوي وحده لا يثبت التسرب.</p>
<table><tr><th>الخطر</th><th>الحارس</th></tr><tr><td>تسرب الهدف</td><td>حذف الحقول التي تصف النتيجة اللاحقة</td></tr><tr><td>تسرب المعالجة</td><td>تعلم التعويض من صفوف التدريب فقط</td></tr><tr><td>تسرب الزمن</td><td>التدريب على الطلبات التي نضجت أهدافها قبل بدء التحقق</td></tr><tr><td>تسرب العميل</td><td>استبعاد عملاء التحقق من تدريب الطية نفسها</td></tr><tr><td>تسرب التكرار</td><td>حذف الأحداث المتطابقة ورفض النسخ المتعارضة</td></tr></table>
<p>تُقرأ نسخة التدريب المتسربة لشرح التدقيق. لا تُستخدم بيانات التحدي في اختيار النموذج أو مقارنته في هذا اللاب.</p>
</td>
</tr>
</table>

In [2]:
# Use application-time availability to decide which fields can enter training.
import json, platform, zipfile
from dataclasses import asdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from data_checks import validate_frame
from day2_validation import (ValidationConfig, BASE_PARAMS, leakage_audit,
    deduplicate_applications, make_time_group_split, reserve_tuning_pool,
    tune_on_reserved_pool, negative_controls, evaluate_outer, summarize_folds, reflection_check)

contract = json.loads((ROOT / "data/data_contract.json").read_text(encoding="utf-8"))
dictionary = pd.read_csv(ROOT / "data/feature_dictionary.csv")
dirty = pd.read_csv(ROOT / "data/tamweel_dirty.csv")
validate_frame(dirty, contract, "dirty")
audit = leakage_audit(dirty, dictionary)
assert not audit.action.eq("BLOCK_UNKNOWN").any(), "Review unknown fields before training."
leaked_features = audit.loc[audit.action.eq("DROP_POST_OUTCOME"), "feature"].tolist()
clean, duplicates = deduplicate_applications(dirty.drop(columns=leaked_features))
validate_frame(clean, contract, "train")
target = contract["target"]["name"]
FAST_MODE, FULL_MODE = True, False
assert FAST_MODE != FULL_MODE, "Choose exactly one mode."
config = ValidationConfig(max_trees=200 if FAST_MODE else 600, trials=8, timeout_seconds=120)
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
display(audit)
display(pd.DataFrame([{**duplicates, "predictors": int(audit.action.eq("KEEP").sum()),
                       "positive_rate": clean[target].mean()}]))
print("Explain which two fields you removed and when their information becomes available.")

Matplotlib is building the font cache; this may take a moment.


,feature,role,available_at,action
0,application_id,id,application_date,EXCLUDE_METADATA_OR_TARGET
1,customer_id,id,application_date,EXCLUDE_METADATA_OR_TARGET
2,application_date,split,application_date,EXCLUDE_METADATA_OR_TARGET
3,age,predictor,application_date,KEEP
4,income_sar,predictor,application_date,KEEP
5,loan_amount_sar,predictor,application_date,KEEP
6,tenor_months,predictor,application_date,KEEP
7,bureau_score,predictor,application_date,KEEP
8,dti,predictor,application_date,KEEP
9,months_employed,predictor,application_date,KEEP


,input_rows,retained_rows,duplicate_rows_removed,predictors,positive_rate
0,10000,10000,0,22,0.0789


Explain which two fields you removed and when their information becomes available.


<table style="width:100%; border-collapse:collapse;">
<tr>
<td width="50%" valign="top" dir="ltr" style="padding:14px; border:1px solid #d0d7de;">
<h2>3. Freeze time, customer and label-maturity roles</h2>
<p>The notebook creates three consecutive forward-validation windows. For each fold, training rows must satisfy:</p>
<p><strong>application date + 90 days &lt; validation start</strong></p>
<p>The strict inequality intentionally excludes labels that mature exactly on the boundary. Every customer appearing in a fold's validation period is removed from that fold's training rows.</p>
<ul><li>The design evaluates later requests from customers unseen in the corresponding training fold.</li><li>A customer may appear in more than one validation period; the guarantee is zero train–validation overlap within each fold.</li><li>Earlier rows remain a warm-up period. They must not receive fitted-training predictions disguised as OOF evidence.</li><li>Exact duplicate events are removed before splitting. Conflicting copies stop execution.</li></ul>
<p>Inspect the fold-audit table and figure before interpreting any performance metric.</p>
</td>
<td width="50%" valign="top" dir="rtl" style="padding:14px; border:1px solid #d0d7de;">
<h2>3. جمّد أدوار الزمن والعملاء ونضج الهدف</h2>
<p>ينشئ الدفتر ثلاث فترات تحقق زمنية متتابعة. يجب أن تحقق صفوف تدريب كل طية الشرط الآتي:</p>
<p><strong>تاريخ الطلب + 90 يومًا &lt; بداية التحقق</strong></p>
<p>تستبعد المتباينة الصارمة عمدًا الأهداف التي تنضج في يوم الحد نفسه. ويُحذف من تدريب كل طية كل عميل يظهر في فترة تحقق تلك الطية.</p>
<ul><li>يقيس التصميم طلبات لاحقة من عملاء لم يظهروا في تدريب الطية المقابلة.</li><li>قد يظهر العميل في أكثر من فترة تحقق؛ والضمان هو عدم وجود تداخل بين التدريب والتحقق داخل الطية نفسها.</li><li>تبقى الصفوف المبكرة فترة تمهيد. لا تمنحها تنبؤات تدريب ثم تعرضها بوصفها OOF.</li><li>تُحذف الأحداث المتطابقة قبل التقسيم، بينما توقف النسخ المتعارضة التنفيذ.</li></ul>
<p>افحص جدول تدقيق الطيات والرسم قبل تفسير أي مقياس أداء.</p>
</td>
</tr>
</table>

In [3]:
# Check the customer separation and label cutoff in each fold.
folds = make_time_group_split(clean, horizon_days=contract["target"]["horizon_days"])
fold_audit = pd.DataFrame([fold["audit"] for fold in folds])
for fold in folds:
    tr, va = clean.loc[fold["train"]], clean.loc[fold["valid"]]
    assert not set(tr.customer_id) & set(va.customer_id)
    assert (pd.to_datetime(tr.application_date) + pd.Timedelta(days=90) <
            pd.Timestamp(fold["audit"]["validation_start"])).all()
fold_audit["validation_positive_rate"] = fold_audit.validation_positives / fold_audit.validation_rows
display(fold_audit)

plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
                     "figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.2})
fig, ax = plt.subplots(figsize=(9, 3.6), constrained_layout=True)
x = np.arange(len(fold_audit))
ax.bar(x - .18, fold_audit.train_rows, .36, label="Mature training, customers purged", color="#2667A0")
ax.bar(x + .18, fold_audit.validation_rows, .36, label="Later validation", color="#BA5A17")
ax.set(xticks=x, xticklabels=fold_audit.validation_start, ylabel="Requests", xlabel="Validation starts",
       title="Three forward folds · zero shared customers within each fold")
ax.set_ylim(0, fold_audit.train_rows.max() * 1.35)
ax.legend(fontsize=9)
fig.savefig(ARTIFACTS / "day2_fold_sizes.png", dpi=160)
plt.show()

,fold,validation_start,validation_end_exclusive,train_rows,validation_rows,train_positives,validation_positives,immature_past_rows_removed,customer_overlap_rows_removed,shared_customers,latest_training_label_available,validation_positive_rate
0,1,2023-07-01,2024-01-01,3223,1632,274,110,826,912,0,2023-06-30,0.067402
1,2,2024-01-01,2024-07-01,4460,1674,353,135,785,1348,0,2023-12-31,0.080645
2,3,2024-07-01,2025-01-01,5731,1733,465,139,861,1675,0,2024-06-30,0.080208


/var/folders/mf/f8kr4l7d1rb3vy03cpjyz45r0000gp/T/ipykernel_12767/177879109.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


<table style="width:100%; border-collapse:collapse;">
<tr>
<td width="50%" valign="top" dir="ltr" style="padding:14px; border:1px solid #d0d7de;">
<h2>4. Reserve search before reading outer validation</h2>
<p>Hyperparameters are selected once from a historical cohort that is mature before the first outer-validation window. All customers used in any of the three outer-validation periods are excluded from this search cohort.</p>
<p>Inside the reserved cohort, three forward folds also separate customers. Each fold contains an inner stopping split: preprocessing and trees are learned on inner-fit rows, the tree count is selected on inner-stop log loss, and the model is then refit on the full fold-training rows.</p>
<ul><li>Outer-validation rows never select imputation, tree count or hyperparameters.</li><li>Future target values are not used to define search parameters.</li><li>The released data produces a relatively small search cohort, with few positive examples in some inner partitions. Record this limitation.</li></ul>
<p>The best search AP is a model-selection statistic. It is not an estimate of future outer performance.</p>
</td>
<td width="50%" valign="top" dir="rtl" style="padding:14px; border:1px solid #d0d7de;">
<h2>4. احجز البحث قبل قراءة التحقق الخارجي</h2>
<p>تُختار المعاملات مرة واحدة من مجموعة تاريخية نضجت أهدافها قبل أول فترة تحقق خارجي. ويُستبعد من مجموعة البحث جميع العملاء المستخدمين في فترات التحقق الخارجي الثلاث.</p>
<p>داخل المجموعة المحجوزة توجد ثلاث طيات زمنية تفصل العملاء أيضًا. تتضمن كل طية جزءًا داخليًا للإيقاف: تتعلم المعالجة والأشجار من صفوف inner fit، ويُختار عدد الأشجار من inner stop باستخدام log loss، ثم يُعاد التدريب على صفوف تدريب الطية كاملة.</p>
<ul><li>لا تختار صفوف التحقق الخارجي التعويض أو عدد الأشجار أو معاملات البحث.</li><li>لا تُستخدم أهداف المستقبل في تحديد إعدادات البحث.</li><li>تنتج البيانات المنشورة مجموعة بحث صغيرة نسبيًا، مع أمثلة موجبة قليلة في بعض الأجزاء الداخلية. وثّق هذا القيد.</li></ul>
<p>أفضل AP داخل البحث إحصائية لاختيار النموذج، وليست تقديرًا للأداء الخارجي المستقبلي.</p>
</td>
</tr>
</table>

In [4]:
# Keep every outer-validation customer out of the search pool.
pool = reserve_tuning_pool(clean, folds)
outer_ids = np.concatenate([fold["valid"] for fold in folds])
assert not set(pool.customer_id) & set(clean.loc[outer_ids, "customer_id"])
assert (pd.to_datetime(pool.application_date) + pd.Timedelta(days=90) < pd.Timestamp("2023-07-01")).all()
display(pd.DataFrame([{"search_rows": len(pool), "search_positives": int(pool[target].sum()),
    "search_positive_rate": pool[target].mean(), "latest_application": pool.application_date.max(),
    "outer_validation_rows": len(outer_ids)}]))

,search_rows,search_positives,search_positive_rate,latest_application,outer_validation_rows
0,1935,176,0.090956,2023-04-01,5039


<table style="width:100%; border-collapse:collapse;">
<tr>
<td width="50%" valign="top" dir="ltr" style="padding:14px; border:1px solid #d0d7de;">
<h2>5. Run a bounded Optuna search</h2>
<p>The objective is mean <strong>Average Precision (AP)</strong> across three internal forward folds. Search changes only <code>learning_rate</code>, <code>num_leaves</code> and <code>min_child_samples</code>.</p>
<ul><li>Seed: 211.</li><li>At most eight sequential trials.</li><li>At most 120 seconds for search, with cooperative stopping that can finish slightly after the boundary.</li><li><code>COMPLETE</code>: all three fold scores were calculated.</li><li><code>PRUNED</code>: the trial stopped early.</li><li><code>FAIL</code>: the attempt did not produce a complete objective; partial values are not a completed mean.</li></ul>
<p>If no trial completes, the notebook shows a stored <code>EDUCATIONAL_EXAMPLE</code> for discussion only and reports <code>TUNING_REQUIRED</code>. The example is never copied into your live results or submission. Retry later in a fresh free-CPU session rather than purchasing compute.</p>
</td>
<td width="50%" valign="top" dir="rtl" style="padding:14px; border:1px solid #d0d7de;">
<h2>5. نفّذ بحث Optuna محدودًا</h2>
<p>الهدف هو متوسط <strong>Average Precision (AP)</strong> عبر ثلاث طيات زمنية داخلية. يغيّر البحث فقط <code>learning_rate</code> و<code>num_leaves</code> و<code>min_child_samples</code>.</p>
<ul><li>البذرة: 211.</li><li>حد أقصى قدره ثماني تجارب متتابعة.</li><li>حد أقصى قدره 120 ثانية للبحث، مع إيقاف تعاوني قد ينتهي بعد الحد بقليل.</li><li><code>COMPLETE</code>: اكتملت درجات الطيات الثلاث.</li><li><code>PRUNED</code>: أوقفت التجربة مبكرًا.</li><li><code>FAIL</code>: لم تنتج المحاولة هدفًا مكتملًا؛ ولا تُعامل القيم الجزئية بوصفها متوسطًا مكتملًا.</li></ul>
<p>إذا لم تكتمل أي تجربة، يعرض الدفتر <code>EDUCATIONAL_EXAMPLE</code> محفوظًا للنقاش فقط ويظهر <code>TUNING_REQUIRED</code>. لا يُنسخ المثال إلى نتائجك الفعلية أو تسليمك. أعد المحاولة لاحقًا في جلسة CPU مجانية جديدة بدل شراء موارد حوسبة.</p>
</td>
</tr>
</table>

In [5]:
# Freeze the best completed trial before scoring the outer folds.
search, history = tune_on_reserved_pool(pool, contract, config)
has_tuned = search["best_params"] is not None
display(pd.DataFrame([{key: search[key] for key in ["status", "completed_trials", "attempted_trials",
    "elapsed_seconds", "stopping_reason", "best_tuning_ap"]}]))
display(pd.DataFrame(search["fold_audits"]))
display(history)
if not has_tuned:
    example = pd.read_csv(ROOT / "data/day2_optuna_example.csv")
    example_info = json.loads((ROOT / "data/day2_optuna_example.json").read_text(encoding="utf-8"))
    assert example_info["source"] == "EDUCATIONAL_EXAMPLE"
    print("EDUCATIONAL_EXAMPLE — discussion only; not your run or submission. TUNING_REQUIRED.")
    display(example)
else:
    print("Frozen search parameters:", search["best_params"])

fig, ax = plt.subplots(figsize=(9, 3.5), constrained_layout=True)
completed = history[history.state.eq("COMPLETE")]
if len(completed):
    ax.plot(completed.trial + 1, completed.mean_ap, marker="o", color="#2667A0", label="Completed trial mean AP")
    ax.step(completed.trial + 1, completed.mean_ap.cummax(), where="post", color="#BA5A17",
            linestyle="--", label="Best completed so far")
    ax.legend(fontsize=9)
else:
    ax.text(.5, .5, "No completed live trials — rerun needed", transform=ax.transAxes, ha="center")
ax.set(xlabel="Trial (1-based)", ylabel="Search AP", ylim=(0, 1), xticks=range(1, config.trials+1),
       title="Live search only · three internal folds · not outer performance")
fig.savefig(ARTIFACTS / "day2_search.png", dpi=160)
plt.show()

,status,completed_trials,attempted_trials,elapsed_seconds,stopping_reason,best_tuning_ap
0,COMPLETE,8,8,1.383538,TRIAL_LIMIT,0.333269


,fold,validation_start,validation_end_exclusive,train_rows,validation_rows,train_positives,validation_positives,immature_past_rows_removed,customer_overlap_rows_removed,shared_customers,latest_training_label_available
0,1,2022-10-01,2023-01-01,703,381,52,42,351,99,0,2022-09-30
1,2,2023-01-01,2023-03-01,1070,271,83,27,373,91,0,2022-12-31
2,3,2023-03-01,2023-04-02,1355,130,118,15,389,61,0,2023-02-28


,trial,state,mean_ap,seconds,learning_rate,num_leaves,min_child_samples,fold_1_ap,fold_2_ap,fold_3_ap
0,0,COMPLETE,0.316524,0.126500,0.085161,15,10,0.319910,0.320935,0.308727
1,1,COMPLETE,0.333269,0.159404,0.085305,31,40,0.391154,0.349373,0.259279
2,2,COMPLETE,0.308752,0.197017,0.034906,31,40,0.404534,0.271448,0.250276
3,3,COMPLETE,0.320550,0.235796,0.040259,31,20,0.384913,0.316815,0.259923
4,4,COMPLETE,0.327798,0.159205,0.062862,15,40,0.403532,0.310814,0.269049
5,5,COMPLETE,0.319093,0.139191,0.037608,7,10,0.365309,0.299275,0.292694
6,6,COMPLETE,0.318537,0.179028,0.039240,15,40,0.414490,0.284021,0.257101
7,7,COMPLETE,0.319024,0.179271,0.062020,31,20,0.376677,0.322038,0.258357


Frozen search parameters: {'learning_rate': 0.08530488562858765, 'num_leaves': 31, 'min_child_samples': 40}


/var/folders/mf/f8kr4l7d1rb3vy03cpjyz45r0000gp/T/ipykernel_12767/2304162047.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


<table style="width:100%; border-collapse:collapse;">
<tr>
<td width="50%" valign="top" dir="ltr" style="padding:14px; border:1px solid #d0d7de;">
<h2>6. Compare only after search is frozen</h2>
<p>Every protocol uses LightGBM, but the data protocol changes:</p>
<table><tr><th>Protocol</th><th>Purpose</th></tr><tr><td>Leaky random control</td><td>Deliberately uses two post-outcome fields and all-row imputation; unsafe negative control</td></tr><tr><td>Clean random control</td><td>Uses eligible fields and training-only imputation but still mixes time and customers; unsafe for decisions</td></tr><tr><td>Honest fixed</td><td>Uses forward mature-label, customer-purged folds with the fixed base parameters and 80 trees</td></tr><tr><td>Honest reserved search</td><td>Uses the same honest outer folds with parameters frozen on the separate search cohort and internal tree selection</td></tr></table>
<p>The random protocols cover 10,000 requests; the time-based protocols cover 5,039 later eligible requests and use different training sizes. Gaps are descriptive and combine period, population and data-availability differences. They are not isolated causal estimates of leakage.</p>
</td>
<td width="50%" valign="top" dir="rtl" style="padding:14px; border:1px solid #d0d7de;">
<h2>6. قارن بعد تجميد البحث</h2>
<p>تستخدم جميع البروتوكولات LightGBM، لكن بروتوكول البيانات يتغير:</p>
<table><tr><th>البروتوكول</th><th>الغرض</th></tr><tr><td>Leaky random control</td><td>يستخدم عمدًا خاصيتين بعد النتيجة وتعويضًا من كامل البيانات؛ ضابط سلبي غير آمن</td></tr><tr><td>Clean random control</td><td>يستخدم الخصائص المؤهلة وتعويض التدريب فقط، لكنه يخلط الزمن والعملاء؛ غير صالح للقرار</td></tr><tr><td>Honest fixed</td><td>يستخدم طيات زمنية ناضجة الأهداف ومستبعدة العملاء مع الإعداد الأساسي الثابت و80 شجرة</td></tr><tr><td>Honest reserved search</td><td>يستخدم الطيات الخارجية الصادقة نفسها مع معاملات مجمدة من مجموعة البحث واختيار داخلي لعدد الأشجار</td></tr></table>
<p>تغطي البروتوكولات العشوائية 10,000 طلب، بينما تغطي البروتوكولات الزمنية 5,039 طلبًا لاحقًا مؤهلًا وبأحجام تدريب مختلفة. الفروق وصفية وتجمع اختلافات الفترة والسكان والبيانات المتاحة، وليست تقديرًا سببيًا مستقلًا للتسرب.</p>
</td>
</tr>
</table>

In [6]:
# Compare fixed and tuned LightGBM on the same time-and-customer folds.
controls = negative_controls(clean, dirty, contract, config)
fixed_report, fixed_oof, fixed_origin = evaluate_outer(
    clean, contract, folds, BASE_PARAMS, config, "clean_time_group_fixed", fixed_trees=80)
reports, predictions, origins = [controls, fixed_report], [fixed_oof], fixed_origin
if has_tuned:
    tuned_report, tuned_oof, tuned_origin = evaluate_outer(
        clean, contract, folds, search["best_params"], config, "clean_time_group_tuned")
    reports.append(tuned_report); predictions.append(tuned_oof); origins += tuned_origin
validation_report = pd.concat(reports, ignore_index=True)
summary = summarize_folds(validation_report)
oof = pd.concat(predictions, ignore_index=True)
assert not oof.duplicated(["scheme", "application_id"]).any()
display(summary.round(4))
display(validation_report[["scheme", "fold", "roc_auc", "average_precision", "train_rows",
    "validation_rows", "selected_trees", "shared_customers", "prediction_source"]].round(4))
gaps = summary.set_index("scheme")
print("Observed AP gap, leaky minus clean random:",
      round(gaps.loc["leaky_random_control", "ap_mean"] - gaps.loc["clean_random_control", "ap_mean"], 4))
print("Observed AP gap, clean random minus honest fixed:",
      round(gaps.loc["clean_random_control", "ap_mean"] - gaps.loc["clean_time_group_fixed", "ap_mean"], 4))

,scheme,folds,validation_rows,roc_auc_mean,roc_auc_sd,ap_mean,ap_sd
0,leaky_random_control,3,10000,0.9999,0.0002,0.9988,0.0014
1,clean_random_control,3,10000,0.8010,0.0200,0.3110,0.0293
2,clean_time_group_fixed,3,5039,0.7976,0.0206,0.3153,0.0426
3,clean_time_group_tuned,3,5039,0.7855,0.0232,0.3133,0.0259


,scheme,fold,roc_auc,average_precision,train_rows,validation_rows,selected_trees,shared_customers,prediction_source
0,leaky_random_control,1,0.9996,0.9973,6666,3334,80,1713,NEGATIVE_CONTROL_NOT_FOR_DECISIONS
1,clean_random_control,1,0.7807,0.2776,6666,3334,80,1713,NEGATIVE_CONTROL_NOT_FOR_DECISIONS
2,leaky_random_control,2,0.9999,0.9992,6667,3333,80,1675,NEGATIVE_CONTROL_NOT_FOR_DECISIONS
3,clean_random_control,2,0.8018,0.3232,6667,3333,80,1675,NEGATIVE_CONTROL_NOT_FOR_DECISIONS
4,leaky_random_control,3,1.0000,1.0000,6667,3333,80,1656,NEGATIVE_CONTROL_NOT_FOR_DECISIONS
5,clean_random_control,3,0.8206,0.3321,6667,3333,80,1656,NEGATIVE_CONTROL_NOT_FOR_DECISIONS
6,clean_time_group_fixed,1,0.7948,0.3395,3223,1632,80,0,LIVE_OUT_OF_FOLD
7,clean_time_group_fixed,2,0.8194,0.3403,4460,1674,80,0,LIVE_OUT_OF_FOLD
8,clean_time_group_fixed,3,0.7786,0.2661,5731,1733,80,0,LIVE_OUT_OF_FOLD
9,clean_time_group_tuned,1,0.7826,0.3356,3223,1632,24,0,LIVE_OUT_OF_FOLD


Observed AP gap, leaky minus clean random: 0.6879
Observed AP gap, clean random minus honest fixed: -0.0044


<table style="width:100%; border-collapse:collapse;">
<tr>
<td width="50%" valign="top" dir="ltr" style="padding:14px; border:1px solid #d0d7de;">
<h2>Read mean and variation together</h2>
<p>The reported score is the unweighted mean across three folds. Error bars show the sample standard deviation with <code>ddof=1</code>.</p>
<ul><li>The folds are related time periods, not three independent experiments.</li><li>The error bar is not a confidence interval and not a significance test.</li><li>AP means <strong>Average Precision</strong>, not trapezoidal area under the precision–recall curve.</li><li>Read fold size, positive rate and coverage beside the headline metrics.</li></ul>
<p>Selected tree counts or the last decimal places can differ across systems even with pinned versions. Do not repeat search because an outer-validation number is disappointing; repeated selection on outer folds converts them into development data.</p>
</td>
<td width="50%" valign="top" dir="rtl" style="padding:14px; border:1px solid #d0d7de;">
<h2>اقرأ المتوسط والتشتت معًا</h2>
<p>الدرجة المعروضة هي المتوسط غير الموزون للطيات الثلاث. وتمثل أشرطة الخطأ الانحراف المعياري العيّني باستخدام <code>ddof=1</code>.</p>
<ul><li>الطيات فترات زمنية مترابطة وليست ثلاث تجارب مستقلة.</li><li>شريط الخطأ ليس فترة ثقة ولا اختبار دلالة.</li><li>يشير AP إلى <strong>Average Precision</strong> وليس المساحة شبه المنحرفة تحت منحنى precision–recall.</li><li>اقرأ حجم الطية ونسبة الفئة الموجبة والتغطية بجانب المقاييس الرئيسية.</li></ul>
<p>قد يختلف عدد الأشجار المختار أو آخر المنازل العشرية بين الأنظمة رغم تثبيت الإصدارات. لا تُعد البحث لأن رقم التحقق الخارجي لم يعجبك؛ فالاختيار المتكرر على الطيات الخارجية يحولها إلى بيانات تطوير.</p>
</td>
</tr>
</table>

In [7]:
# Show the fold-to-fold spread beside each mean.
labels = {"leaky_random_control": "Leaky random (unsafe)", "clean_random_control": "Clean random (unsafe)",
          "clean_time_group_fixed": "Honest, fixed 80 trees", "clean_time_group_tuned": "Honest, reserved search"}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), constrained_layout=True)
y = np.arange(len(summary))
for ax, mean_col, sd_col, title in zip(axes, ["roc_auc_mean", "ap_mean"], ["roc_auc_sd", "ap_sd"], ["ROC-AUC", "Average precision"]):
    ax.errorbar(summary[mean_col], y, xerr=summary[sd_col], fmt="o", capsize=5, color="#2667A0")
    ax.set(yticks=y, yticklabels=[labels[s] for s in summary.scheme], xlim=(0, 1.03),
           xlabel="Mean ± sample SD (not a confidence interval)", title=title)
    ax.invert_yaxis()
    for row, value in enumerate(summary[mean_col]):
        ax.annotate(f"{value:.3f}", (value, row), xytext=(-6, 10), textcoords="offset points", ha="right", fontsize=9)
fig.suptitle("Three folds per protocol · populations differ · descriptive comparison")
fig.savefig(ARTIFACTS / "day2_validation_comparison.png", dpi=160)
plt.show()

/var/folders/mf/f8kr4l7d1rb3vy03cpjyz45r0000gp/T/ipykernel_12767/2936520195.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


<table style="width:100%; border-collapse:collapse;">
<tr>
<td width="50%" valign="top" dir="ltr" style="padding:14px; border:1px solid #d0d7de;">
<h2>7. Verify out-of-fold coverage</h2>
<p>Every saved probability is generated for an application outside the corresponding fold's training rows after its internal choices are fixed.</p>
<ul><li>Each successful honest scheme produces 5,039 OOF probabilities.</li><li>The remaining 4,961 applications are warm-up rows without OOF predictions.</li><li>Coverage is 50.39% of all clean training rows and 100% of requests eligible from 2023-07-01 onward.</li><li>Do not fill warm-up rows with training predictions or artificial probabilities.</li><li>Do not export the leaky-control predictions for later decision work.</li></ul>
<p>This OOF set is not an independent final test and does not guarantee fairness or calibration. Preserve each row's role so later decisions use declared coverage.</p>
</td>
<td width="50%" valign="top" dir="rtl" style="padding:14px; border:1px solid #d0d7de;">
<h2>7. تحقّق من تغطية OOF</h2>
<p>يُنتج كل احتمال محفوظ لطلب خارج صفوف تدريب الطية المقابلة بعد تثبيت خياراتها الداخلية.</p>
<ul><li>ينتج كل إعداد صادق ناجح 5,039 احتمال OOF.</li><li>تبقى الطلبات البالغ عددها 4,961 صفوف تمهيد بلا تنبؤات OOF.</li><li>تبلغ التغطية 50.39% من جميع صفوف التدريب النظيفة و100% من الطلبات المؤهلة بدءًا من 2023-07-01.</li><li>لا تملأ صفوف التمهيد بتنبؤات التدريب أو احتمالات مصطنعة.</li><li>لا تصدّر تنبؤات الضابط المتسرب لأعمال القرار اللاحقة.</li></ul>
<p>لا تمثل مجموعة OOF اختبارًا نهائيًا مستقلًا، ولا تضمن العدالة أو المعايرة. احتفظ بدور كل صف حتى تُبنى القرارات اللاحقة على تغطية معلنة.</p>
</td>
</tr>
</table>

In [8]:
# Check that each honest model scores exactly the eligible application IDs.
coverage = clean[["application_id", "application_date"]].copy()
coverage["role"], coverage["fold"] = "warmup_no_oof", 0
for fold in folds:
    coverage.loc[fold["valid"], "role"] = "eligible_oof"
    coverage.loc[fold["valid"], "fold"] = fold["audit"]["fold"]
eligible_ids = set(coverage.loc[coverage.role.eq("eligible_oof"), "application_id"])
for scheme, group in oof.groupby("scheme"):
    assert set(group.application_id) == eligible_ids
    assert group.probability.between(0, 1).all()
display(pd.DataFrame([{"all_rows": len(clean), "eligible_rows": len(eligible_ids),
    "warmup_without_oof": len(clean)-len(eligible_ids), "whole_data_coverage": len(eligible_ids)/len(clean),
    "eligible_coverage": 1.0, "live_honest_schemes": oof.scheme.nunique()}]))

,all_rows,eligible_rows,warmup_without_oof,whole_data_coverage,eligible_coverage,live_honest_schemes
0,10000,5039,4961,0.5039,1.0,2


<table style="width:100%; border-collapse:collapse;">
<tr>
<td width="50%" valign="top" dir="ltr" style="padding:14px; border:1px solid #d0d7de;">
<h2>8. Write your own validation argument</h2>
<p>Complete the shared response dictionary in your own words, then rerun the response and export cells:</p>
<ol><li>Name one leaked field and explain why its information was unavailable at application time.</li><li>Justify time ordering, customer separation and the strict 90-day maturity rule.</li><li>Cite two numerical gaps from your validation table and explain why they do not prove universal superiority.</li><li>Explain why imputation is learned inside training and what the inner stopping set controls.</li><li>State one search-budget limitation and one OOF-coverage or fold-dependence limitation.</li></ol>
<p>The checkpoint detects missing fields only. It does not write your answer, judge correctness, award a grade or prove authorship.</p>
</td>
<td width="50%" valign="top" dir="rtl" style="padding:14px; border:1px solid #d0d7de;">
<h2>8. اكتب حجتك الخاصة للتحقق</h2>
<p>أكمل قاموس الإجابات المشترك بصياغتك أنت، ثم أعد تشغيل خلية الإجابات وخلية التصدير:</p>
<ol><li>اذكر خاصية متسربة، واشرح لماذا لم تكن معلومتها متاحة وقت تقديم الطلب.</li><li>برر ترتيب الزمن وفصل العملاء وقاعدة نضج الهدف الصارمة لمدة 90 يومًا.</li><li>استشهد بفجوتين رقميتين من جدول التحقق، واشرح لماذا لا تثبتان تفوقًا عامًا.</li><li>اشرح لماذا يُتعلم التعويض داخل التدريب وما الذي يتحكم فيه جزء الإيقاف الداخلي.</li><li>اذكر قيدًا في ميزانية البحث وقيدًا في تغطية OOF أو اعتماد الطيات.</li></ol>
<p>يفحص الحارس اكتمال الخانات فقط؛ لا يكتب إجابتك، ولا يحكم على صحتها، ولا يمنح درجة، ولا يثبت الملكية.</p>
</td>
</tr>
</table>

In [9]:
responses = {'leak_example': 'collection_calls counted follow-up contacts after the application. The '
                 'application-time model could not know that value. days_past_due_60 was excluded for '
                 'the same timing reason.',
 'split_reason': 'Validate on later applications and hold those customers out of training for that '
                 'fold. Include earlier rows only after the full 90-day outcome window has closed, so '
                 'training only uses labels it could have known.',
 'observed_gap': 'The leaky and clean random controls had mean AP 0.9988 and 0.3110, a difference of '
                 'about 0.688. Within the same three forward folds, fixed LightGBM had 0.3153 AP versus '
                 '0.3133 tuned. These three windows and their customers limit generalization.',
 'imputer_reason': 'Training-fold medians avoid using evaluation data to learn replacement values. The '
                   'inner stopping set selects the number of trees within development, leaving the '
                   'outer period for evaluation.',
 'limitations': 'Optuna tested eight trials on 1,935 reserved historical rows. The 5,039 OOF '
                'predictions cover 50.39% of all applications, and the expanding folds share earlier '
                'training history.'}
checkpoint = reflection_check(responses)
print(checkpoint["status"])
print("Missing:", checkpoint["missing"])

READY_FOR_REVIEW
Missing: []


<table style="width:100%; border-collapse:collapse;">
<tr>
<td width="50%" valign="top" dir="ltr" style="padding:14px; border:1px solid #d0d7de;">
<h2>9. Export evidence and preserve provenance</h2>
<p>The export cell writes the validation tables, search record, audits, OOF predictions and coverage, provenance, reflection, run metadata, three figures and <code>environment.json</code>, then creates <code>day2_artifacts.zip</code>.</p>
<p>Download the ZIP from <strong>Files → tamweel → artifacts</strong>, extract it and upload the actual files to your repository's <code>artifacts/</code> directory. Save the executed notebook as <code>notebooks/02_validation_tuning.ipynb</code>.</p>
<ul><li><code>TECHNICAL_READY</code>: execution and at least one live tuning trial completed.</li><li><code>TUNING_REQUIRED</code>: no live trial completed; rerun is required.</li><li><code>LEARNER_WORK_REQUIRED</code>: technical work exists, but your written fields are incomplete.</li><li><code>READY_FOR_REVIEW</code>: required fields are present; this is not a grade or correctness guarantee.</li></ul>
<p>Saving again replaces this day's artifact names. Preserve a copy before rerunning. Never upload credentials, access tokens or personal data.</p>
</td>
<td width="50%" valign="top" dir="rtl" style="padding:14px; border:1px solid #d0d7de;">
<h2>9. صدّر الأدلة واحتفظ بمصدرها</h2>
<p>تكتب خلية التصدير جداول التحقق وسجل البحث والتدقيقات وتنبؤات OOF وتغطيتها ومصدر التدريب والتفسير وبيانات التشغيل وثلاثة رسوم و<code>environment.json</code>، ثم تنشئ <code>day2_artifacts.zip</code>.</p>
<p>نزّل الحزمة من <strong>Files → tamweel → artifacts</strong>، وافكها، وارفع الملفات الفعلية إلى مجلد <code>artifacts/</code> في مستودعك. واحفظ الدفتر المنفذ باسم <code>notebooks/02_validation_tuning.ipynb</code>.</p>
<ul><li><code>TECHNICAL_READY</code>: نجح التنفيذ واكتملت تجربة ضبط فعلية واحدة على الأقل.</li><li><code>TUNING_REQUIRED</code>: لم تكتمل تجربة فعلية؛ يلزم إعادة التشغيل.</li><li><code>LEARNER_WORK_REQUIRED</code>: توجد الأدلة التقنية، لكن حقولك الكتابية غير مكتملة.</li><li><code>READY_FOR_REVIEW</code>: الحقول المطلوبة موجودة؛ ولا تمثل درجة أو ضمانًا لصحة التفسير.</li></ul>
<p>يستبدل الحفظ المتكرر أسماء ملفات هذا اليوم. احتفظ بنسخة قبل إعادة التشغيل. لا ترفع بيانات اعتماد أو رموز وصول أو بيانات شخصية.</p>
</td>
</tr>
</table>

In [10]:
# Export the daily tables and keep the full evidence archive.
checkpoint = reflection_check(responses)
technical_status = "TECHNICAL_READY" if has_tuned else "TUNING_REQUIRED"
csv_outputs = {"validation_report.csv": validation_report, "validation_summary.csv": summary,
    "optuna_results.csv": history, "leakage_audit.csv": audit, "fold_audit.csv": fold_audit,
    "day2_oof_predictions.csv": oof, "day2_oof_coverage.csv": coverage}
for filename, table in csv_outputs.items():
    table.to_csv(ARTIFACTS / filename, index=False)
run = {"technical_status": technical_status, "learner_status": checkpoint["status"],
    "data_revision": COURSE_REVISION, "lab_revision": LAB_REVISION, "support_sha256": SUPPORT_HASHES,
    "data_manifest_sha256": MANIFEST_SHA256,
    "dirty_data_sha256": hashlib.sha256((ROOT / "data/tamweel_dirty.csv").read_bytes()).hexdigest(),
    "config": asdict(config), "mode": "FAST" if FAST_MODE else "FULL", "python": platform.python_version(),
    "duplicates": duplicates, "removed_features": leaked_features,
    "oof_rows_per_scheme": {name: len(group) for name, group in oof.groupby("scheme")},
    "oof_whole_data_coverage": len(eligible_ids)/len(clean), "warmup_rows": len(clean)-len(eligible_ids),
    "search_source": "LIVE", "example_used_for_results": False,
    "challenge_used_for_modeling": False, "grading": "No automatic grade"}
json_outputs = {"best_params.json": search, "day2_provenance.json": origins,
                "day2_reflection.json": {"responses": responses, "checkpoint": checkpoint}, "day2_run.json": run}
for filename, payload in json_outputs.items():
    (ARTIFACTS / filename).write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
files = list(csv_outputs) + list(json_outputs) + ["day2_fold_sizes.png", "day2_search.png",
    "day2_validation_comparison.png", "environment.json"]
with zipfile.ZipFile(ARTIFACTS / "day2_artifacts.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for filename in files:
        assert (ARTIFACTS / filename).is_file(), filename
        archive.write(ARTIFACTS / filename, arcname=filename)
print(technical_status, "|", checkpoint["status"])
print(f"Saved {len(files)} files in day2_artifacts.zip | احفظ الدفتر وملفاتك ثم أكمل تفسيرك")

TECHNICAL_READY | READY_FOR_REVIEW
Saved 15 files in day2_artifacts.zip | احفظ الدفتر وملفاتك ثم أكمل تفسيرك


<table style="width:100%; border-collapse:collapse;">
<tr>
<td width="50%" valign="top" dir="ltr" style="padding:14px; border:1px solid #d0d7de;">
<h2>Completion checkpoint and next step</h2>
<ul><li>You can identify the removed post-outcome fields and explain their availability time.</li><li>Each fold shows zero shared customers and mature training labels before validation starts.</li><li>You can distinguish attempted, completed, pruned and failed search trials.</li><li>You read fold means, sample standard deviations, population differences and OOF coverage together.</li></ul>
<p><strong>Before presenting results:</strong> show one leakage example, point to evidence of customer separation and label maturity, distinguish mean fold AP from pooled AP, and state the result's limits beside the score.</p>
<p><strong>References:</strong> scikit-learn cross-validation and preprocessing-leakage guidance, LightGBM early stopping and Optuna bounded-study documentation linked in the course guide.</p>
<p><strong>Day 3:</strong> use the honest validation foundation to examine simulated decision costs, thresholds and the consequences of false approvals and false declines.</p>
</td>
<td width="50%" valign="top" dir="rtl" style="padding:14px; border:1px solid #d0d7de;">
<h2>نقطة الإكمال والخطوة التالية</h2>
<ul><li>تستطيع تحديد خصائص ما بعد النتيجة المحذوفة وشرح زمن إتاحة معلوماتها.</li><li>تُظهر كل طية صفر عملاء مشتركين ونضج أهداف التدريب قبل بدء التحقق.</li><li>تستطيع التمييز بين محاولات البحث والمكتمل منها والموقوفة والفاشلة.</li><li>تقرأ متوسطات الطيات والانحرافات المعيارية العيّنية واختلاف السكان وتغطية OOF معًا.</li><li>أكملت الإجابات الخمس التي يملكها المتدرب، وحفظت الدفتر المنفذ وحزمة الأدلة خارج الجلسة المؤقتة.</li></ul>
<p><strong>قبل عرض النتائج:</strong> اعرض مثالًا للتسرب، وأشر إلى دليل فصل العملاء ونضج الهدف، وميّز بين متوسط AP عبر الطيات وAP بعد جمع الاحتمالات، وضع حدود النتيجة بجانب الدرجة.</p>
<p><strong>المراجع:</strong> إرشادات scikit-learn للتحقق ومنع تسرب المعالجة، وتوثيق LightGBM للإيقاف المبكر، وتوثيق Optuna للبحث المحدود، وكلها مرتبطة في دليل الدورة.</p>
<p><strong>اليوم الثالث:</strong> استخدم أساس التحقق الصادق لفحص تكاليف القرار التعليمية والعتبات وآثار الموافقات الخاطئة والرفض الخاطئ.</p>
</td>
</tr>
</table>

<table style="width:100%; border-collapse:collapse;">
<tr>
<td width="50%" valign="top" dir="ltr" style="padding:14px; border:1px solid #d0d7de;">
<h2>Daily question · Honest validation</h2>
<p><strong>What leakage risk did you identify, and how did the validation design prevent it?</strong><br><br>Write a short answer under <code>Day 2</code> in the <code>Daily answers | الإجابات اليومية</code> section of your repository README. Then download this executed notebook and the required generated files and upload them on the same day.</p>
</td>
<td width="50%" valign="top" dir="rtl" style="padding:14px; border:1px solid #d0d7de;">
<h2>سؤال اليوم · التحقق الصادق</h2>
<p><strong>ما خطر التسرب الذي اكتشفته، وكيف منعه تصميم التحقق؟</strong><br><br>اكتب إجابة مختصرة تحت <code>Day 2 | اليوم 2</code> في قسم <code>Daily answers | الإجابات اليومية</code> داخل README في مستودعك، ثم نزّل هذا الدفتر بعد التنفيذ وملفات النتائج المطلوبة وارفعها في اليوم نفسه.</p>
</td>
</tr>
</table>

## Daily answer | إجابة اليوم

I found that `collection_calls` counted contacts made after the application. I removed it and `days_past_due_60` because neither was known when the applicant applied. The leaky random control scored 0.9988 mean AP, compared with 0.3110 for the clean random control. The clean run also learned its imputation medians inside training. For the later validation windows, I excluded those customers from training and used only earlier applications whose 90-day outcome window had already closed. All three folds had zero shared customers.